In [1]:
import pandas as pd

import sys
sys.path.insert(0, "../src")
from data_cleaner import DataCleaner

In [2]:
import pickle
# load
with open('../output/checkpoint/1.pkl', 'rb') as f:
    checkpoint = pickle.load(f)


import pickle
# load
with open('../output/checkpoint/preprocessor.pkl', 'rb') as f:
    preprocessor = pickle.load(f)

In [8]:
checkpoint

,estimator,RandomForestC...ndom_state=42)
,param_distributions,"{'max_depth': [15, 16, ...], 'min_samples_leaf': [1], 'min_samples_split': [7, 8, ...], 'n_estimators': [435, 436, ...]}"
,n_iter,10
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,5
,verbose,3
,pre_dispatch,'2*n_jobs'
,random_state,42
,error_score,'raise'


In [3]:
cleaner = DataCleaner()

In [4]:
test_val = pd.read_csv("../data/test_set_values.csv")
id = pd.DataFrame(test_val["id"])

In [5]:
test_val

,id,amount_tsh,date_recorded,funder,gps_height,installer,longitude,latitude,wpt_name,num_private,...,payment_type,water_quality,quality_group,quantity,quantity_group,source,source_type,source_class,waterpoint_type,waterpoint_type_group
0,50785,0.0,2013-02-04,Dmdd,1996,DMDD,35.290799,-4.059696,Dinamu Secondary School,0,...,never pay,soft,good,seasonal,seasonal,rainwater harvesting,rainwater harvesting,surface,other,other
1,51630,0.0,2013-02-04,Government Of Tanzania,1569,DWE,36.656709,-3.309214,Kimnyak,0,...,never pay,soft,good,insufficient,insufficient,spring,spring,groundwater,communal standpipe,communal standpipe
2,17168,0.0,2013-02-01,NaN,1567,NaN,34.767863,-5.004344,Puma Secondary,0,...,never pay,soft,good,insufficient,insufficient,rainwater harvesting,rainwater harvesting,surface,other,other
3,45559,0.0,2013-01-22,Finn Water,267,FINN WATER,38.058046,-9.418672,Kwa Mzee Pange,0,...,unknown,soft,good,dry,dry,shallow well,shallow well,groundwater,other,other
4,49871,500.0,2013-03-27,Bruder,1260,BRUDER,35.006123,-10.950412,Kwa Mzee Turuka,0,...,monthly,soft,good,enough,enough,spring,spring,groundwater,communal standpipe,communal standpipe
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
14845,39307,0.0,2011-02-24,Danida,34,Da,38.852669,-6.582841,Kwambwezi,0,...,never pay,soft,good,enough,enough,river,river/lake,surface,communal standpipe,communal standpipe
14846,18990,1000.0,2011-03-21,Hiap,0,HIAP,37.451633,-5.350428,Bonde La Mkondoa,0,...,annually,salty,salty,insufficient,insufficient,shallow well,shallow well,groundwater,hand pump,hand pump
14847,28749,0.0,2013-03-04,NaN,1476,NaN,34.739804,-4.585587,Bwawani,0,...,never pay,soft,good,insufficient,insufficient,dam,dam,surface,communal standpipe,communal standpipe
14848,33492,0.0,2013-02-18,Germany,998,DWE,35.432732,-10.584159,Kwa John,0,...,never pay,soft,good,insufficient,insufficient,river,river/lake,surface,communal standpipe,communal standpipe


In [6]:
test_val_df = cleaner.extract_recorded_year(dataframe=test_val)
test_val_df = cleaner.fill_extracttion_year(dataframe=test_val_df)
test_val_df = cleaner.get_pump_age(dataframe=test_val_df)
test_val_df = cleaner.extract_feature(dataframe=test_val_df)


test_val_df = preprocessor.fit_transform(test_val_df)

prediction = checkpoint.predict(test_val_df)


# Invert the dictionary
TARGET_MAPPING = {"non functional": 0, "functional needs repair": 1, "functional": 2}
target_mapping_inverted = {v: k for k, v in TARGET_MAPPING.items()}

prediction = list(prediction)
for i,p in enumerate(prediction):
   # label = [key for key, val in TARGET_MAPPING.items() if val == p]
   prediction[i] = target_mapping_inverted.get(p)

pred_df = pd.DataFrame(prediction, columns=["status_group"])
result = id.join(pred_df)
result

,id,status_group
0,50785,functional
1,51630,functional needs repair
2,17168,functional
3,45559,non functional
4,49871,functional
...,...,...
14845,39307,non functional
14846,18990,functional
14847,28749,functional needs repair
14848,33492,functional


In [7]:
result.to_csv('../output/submission/submission.csv', index=False)